In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

In [2]:
# Load the trained model
model = joblib.load('../models/spike_predictor.pkl')
print(f"Loaded model: {type(model).__name__}")

# Load feature names
with open('../models/feature_names.txt', 'r') as f:
    feature_names = [line.strip() for line in f.readlines()]

print(f"Number of features: {len(feature_names)}")
print(f"\nFirst 10 features:")
for i, name in enumerate(feature_names[:10], 1):
    print(f"  {i}. {name}")

Loaded model: RandomForestClassifier
Number of features: 60

First 10 features:
  1. glucose
  2. basal_rate
  3. temp_basal_rate
  4. hypo_event
  5. stress_event
  6. heart_rate
  7. gsr
  8. skin_temp_f
  9. air_temp_f
  10. steps


In [6]:
# Load test features
features_df = pd.read_csv('../data/processed/train_features.csv', parse_dates=['timestamp'])

# Prepare features
exclude_cols = [
    'patient_id', 'timestamp', 'will_spike', 'future_glucose_max',
    'carbs', 'bolus_dose', 'meal_type',
    'exercise_duration', 'exercise_intensity','finger_stick_glucose','glucose_baseline',
       'future_glucose_rise',
    # Exclude all timestamp columns
    'meal_timestamp', 'finger_stick_timestamp', 'bolus_timestamp',
    'exercise_timestamp', 'hypo_timestamp', 'stress_timestamp'
]


X = features_df[[col for col in features_df.columns if col not in exclude_cols]].fillna(0)
y = features_df['will_spike'].fillna(0)

print(f"Test data: {X.shape}")

Test data: (69255, 60)


/var/folders/3p/4rqnf9yd027770h9vdcqrrhw0000gn/T/ipykernel_16928/3962720601.py:2: DtypeWarning: Columns (15,17) have mixed types. Specify dtype option on import or set low_memory=False.
  features_df = pd.read_csv('../data/processed/train_features.csv', parse_dates=['timestamp'])


In [7]:
def generate_explanation(probability, features, feature_names):
    """
    Generate human-readable explanation for spike prediction.
    
    Args:
        probability: Predicted spike probability (0-1)
        features: Feature values (array or Series)
        feature_names: List of feature names
    
    Returns:
        Explanation string
    """
    # Convert to dict for easier access
    feat = dict(zip(feature_names, features))
    
    # Determine risk level
    if probability >= 0.80:
        risk = "VERY HIGH"
    elif probability >= 0.60:
        risk = "HIGH"
    elif probability >= 0.40:
        risk = "MODERATE"
    elif probability >= 0.20:
        risk = "LOW"
    else:
        risk = "VERY LOW"
    
    # Start building explanation
    explanation = f"Risk: {risk} ({probability:.0%} probability of glucose spike above 180 mg/dL in next 2 hours). "
    
    reasons = []
    
    # 1. Current glucose and trend
    glucose_parts = []
    
    # Current level
    if 'current_glucose' in feat and feat['current_glucose'] > 0:
        glucose_parts.append(f"Current glucose is {feat['current_glucose']:.0f} mg/dL")
    elif 'glucose_mean_60min' in feat and feat['glucose_mean_60min'] > 0:
        glucose_parts.append(f"Recent glucose average is {feat['glucose_mean_60min']:.0f} mg/dL")
    elif 'glucose_baseline' in feat and feat['glucose_baseline'] > 0:
        glucose_parts.append(f"Baseline glucose is {feat['glucose_baseline']:.0f} mg/dL")
    
    # Velocity
    if 'glucose_velocity_15min' in feat:
        velocity = feat['glucose_velocity_15min']
        if velocity > 2.0:
            glucose_parts.append(f"with rapidly rising velocity of {velocity:.1f} mg/dL/min")
        elif velocity > 0.8:
            glucose_parts.append(f"with rising velocity of {velocity:.1f} mg/dL/min")
        elif velocity > 0.3:
            glucose_parts.append(f"with slowly rising velocity of {velocity:.1f} mg/dL/min")
        elif velocity < -2.0:
            glucose_parts.append(f"with rapidly falling velocity of {abs(velocity):.1f} mg/dL/min")
        elif velocity < -0.8:
            glucose_parts.append(f"with falling velocity of {abs(velocity):.1f} mg/dL/min")
        elif velocity < -0.3:
            glucose_parts.append(f"with slowly falling velocity of {abs(velocity):.1f} mg/dL/min")
    
    # Delta
    if 'glucose_delta_30min' in feat:
        delta = feat['glucose_delta_30min']
        if abs(delta) > 10:
            if delta > 0:
                glucose_parts.append(f"(up {delta:.0f} mg/dL in last 30 minutes)")
            else:
                glucose_parts.append(f"(down {abs(delta):.0f} mg/dL in last 30 minutes)")
    
    if glucose_parts:
        reasons.append(" ".join(glucose_parts))
    
    # 2. Meal context
    if 'minutes_since_meal' in feat and 'carbs_from_last_meal' in feat:
        minutes = feat['minutes_since_meal']
        carbs = feat['carbs_from_last_meal']
        
        if minutes < 120 and carbs > 0:
            # Time description
            if minutes < 5:
                time_str = "just now"
            elif minutes < 60:
                time_str = f"{int(minutes)} minutes ago"
            else:
                time_str = f"{minutes/60:.1f} hours ago"
            
            # Carb amount description
            if carbs > 60:
                carb_desc = f"large carb intake of {carbs:.0f}g"
            elif carbs > 40:
                carb_desc = f"significant carb intake of {carbs:.0f}g"
            elif carbs > 20:
                carb_desc = f"moderate carb intake of {carbs:.0f}g"
            else:
                carb_desc = f"small carb intake of {carbs:.0f}g"
            
            meal_str = f"Recent {carb_desc} consumed {time_str}"
            
            # Add interaction context
            if feat.get('high_carb_and_rising', 0) == 1:
                meal_str += " with glucose already rising"
            
            reasons.append(meal_str)
    
    # 3. Insulin context
    if 'minutes_since_bolus' in feat and 'dose_from_last_bolus' in feat:
        minutes = feat['minutes_since_bolus']
        dose = feat['dose_from_last_bolus']
        
        if minutes < 120 and dose > 0:
            if minutes < 60:
                time_str = f"{int(minutes)} minutes ago"
            else:
                time_str = f"{minutes/60:.1f} hours ago"
            
            reasons.append(f"Bolus insulin of {dose:.1f}u given {time_str}")
        elif feat.get('meal_no_insulin', 0) == 1:
            reasons.append("No recent insulin coverage for meal")
    
    # 4. Additional factors
    if feat.get('is_exercising', 0) == 1:
        reasons.append("Currently exercising")
    
    if feat.get('glucose_std_30min', 0) > 20:
        reasons.append("High glucose variability detected")
    
    # Combine all reasons
    if reasons:
        explanation += ". ".join(reasons) + "."
    
    return explanation

## 3. Test Explanation Generator

In [11]:
sample_features

,glucose,basal_rate,temp_basal_rate,hypo_event,stress_event,heart_rate,gsr,skin_temp_f,air_temp_f,steps,...,dose_from_last_bolus,bolus_last_60min,bolus_last_120min,meal_no_insulin,high_carb_and_rising,fast_rise_after_meal,accelerating_rise,is_exercising,exercise_last_60min,exercise_last_120min
10183,203.0,1.25,1.25,0.0,0.0,73.0,0.000058,87.2,85.1,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0


In [8]:
# Test on a few samples
n_samples = 5
test_indices = np.random.choice(len(X), n_samples, replace=False)

print("=" * 80)
print("SAMPLE PREDICTIONS WITH EXPLANATIONS")
print("=" * 80)

for i, idx in enumerate(test_indices, 1):
    # Get features
    sample_features = X.iloc[idx:idx+1]
    actual = y.iloc[idx]
    
    # Make prediction
    prediction = model.predict(sample_features)[0]
    probability = model.predict_proba(sample_features)[0, 1]
    
    # Generate explanation
    explanation = generate_explanation(probability, sample_features.values[0], feature_names)
    
    print(f"\n--- Sample {i} ---")
    print(f"Actual: {'Will Spike' if actual == 1 else 'No Spike'}")
    print(f"Predicted: {'Will Spike' if prediction == 1 else 'No Spike'}")
    print(f"Probability: {probability:.2%}")
    print(f"\nExplanation:\n{explanation}")
    print("-" * 80)

SAMPLE PREDICTIONS WITH EXPLANATIONS

--- Sample 1 ---
Actual: No Spike
Predicted: No Spike
Probability: 25.77%

Explanation:
Risk: LOW (26% probability of glucose spike above 180 mg/dL in next 2 hours). Recent glucose average is 108 mg/dL with falling velocity of 1.6 mg/dL/min (down 45 mg/dL in last 30 minutes). Recent small carb intake of 20g consumed just now. No recent insulin coverage for meal.
--------------------------------------------------------------------------------

--- Sample 2 ---
Actual: No Spike
Predicted: No Spike
Probability: 14.88%

Explanation:
Risk: VERY LOW (15% probability of glucose spike above 180 mg/dL in next 2 hours). Recent glucose average is 96 mg/dL with slowly falling velocity of 0.4 mg/dL/min.
--------------------------------------------------------------------------------

--- Sample 3 ---
Actual: Will Spike
Predicted: Will Spike
Probability: 99.86%

Explanation:
Risk: VERY HIGH (100% probability of glucose spike above 180 mg/dL in next 2 hours). Rec

## 4. Create API Response Format

In [ ]:
def format_api_response(user_id, prediction, probability, explanation):
    """
    Format prediction response for API.
    
    Returns:
        Dictionary matching API specification
    """
    return {
        "user_id": user_id,
        "will_spike": bool(prediction),
        "risk_score": round(float(probability), 2),
        "explanation": explanation
    }

# Test the format
import json

idx = test_indices[0]
sample_features = X.iloc[idx:idx+1]
prediction = model.predict(sample_features)[0]
probability = model.predict_proba(sample_features)[0, 1]
explanation = generate_explanation(probability, sample_features.values[0], feature_names)

response = format_api_response(
    user_id="123",
    prediction=prediction,
    probability=probability,
    explanation=explanation
)

print("\nAPI Response Format:")
print(json.dumps(response, indent=2))

## 5. Test High-Risk vs Low-Risk Scenarios

In [ ]:
# Find high-risk examples (probability > 0.8)
predictions = model.predict_proba(X)[:, 1]
high_risk_indices = np.where(predictions > 0.8)[0][:3]
low_risk_indices = np.where(predictions < 0.2)[0][:3]

print("=" * 80)
print("HIGH-RISK EXAMPLES")
print("=" * 80)

for idx in high_risk_indices:
    sample = X.iloc[idx:idx+1]
    prob = predictions[idx]
    explanation = generate_explanation(prob, sample.values[0], feature_names)
    
    print(f"\nProbability: {prob:.2%}")
    print(f"Explanation: {explanation}")
    print("-" * 80)

print("\n" + "=" * 80)
print("LOW-RISK EXAMPLES")
print("=" * 80)

for idx in low_risk_indices:
    sample = X.iloc[idx:idx+1]
    prob = predictions[idx]
    explanation = generate_explanation(prob, sample.values[0], feature_names)
    
    print(f"\nProbability: {prob:.2%}")
    print(f"Explanation: {explanation}")
    print("-" * 80)

## 6. Analyze Feature Contributions

For tree-based models, we can see which features contributed most to each prediction.

In [ ]:
# If model has feature_importances_, show them
if hasattr(model, 'feature_importances_'):
    # Get feature importances
    importances = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features for Predictions:")
    print(importances.head(10).to_string(index=False))
else:
    print("\nModel does not have feature importances (Logistic Regression)")
    
    # For logistic regression, show coefficients
    if hasattr(model, 'coef_'):
        coefficients = pd.DataFrame({
            'feature': feature_names,
            'coefficient': model.coef_[0]
        }).sort_values('coefficient', key=abs, ascending=False)
        
        print("\nTop 10 Features by Coefficient Magnitude:")
        print(coefficients.head(10).to_string(index=False))

## Summary

This notebook demonstrates:

1. **Explanation generation** based on glucose trends, meal timing, and insulin context
2. **API response formatting** matching the assignment specification
3. **Risk level classification** (VERY LOW to VERY HIGH)
4. **Human-readable narratives** explaining the prediction

The `generate_explanation()` function can be integrated directly into the FastAPI endpoint.